# As-Is vs To-Be sector demand curves collapsing into total demand.

Synthetic data built to match a simplify version of the business rules from Natura:
 - 21-day cycle 
 - 3 blocks (instead of 15=3 blocks x 5 sub-blocks)
 - each sector has its own fixed demand-density SHAPE (day-of-window purchase behavior)
   that must be preserved when the sector is reassigned to a different block (As-Is -> To-Be)
 - As-Is: sectors cluster in Block 2 -> ~45% of volume there, sharp peak above capacity
 - To-Be: same shapes, reassigned start day -> interlocking curves, flat total under capacity


In [ ]:
import matplotlib.colors as mcolors
import matplotlib.pyplot as plt
import numpy as np
from mpl_toolkits.mplot3d.art3d import Poly3DCollection


In [ ]:
rng = np.random.default_rng(7)

N_DAYS = 21+15          # full cycle
WINDOW = 21            # each sector's own decision window length (days)
DAYS = np.arange(N_DAYS)
DAYS

## 1. Six sectors, each with a fixed within-window shape (sums to 1)

In [ ]:
def normalize(x):
    x = np.clip(x, 0, None)
    return x / x.sum()

t = np.arange(WINDOW)

shapes = {
    "Setor A (pico inicial)":  normalize(np.exp(-((t - 0.5) ** 2) / (2 * 1.0 ** 2))),
    "Setor B (pico final)":    normalize(np.exp(-((t - 5.5) ** 2) / (2 * 1.0 ** 2))),
    "Setor C (uniforme)":      normalize(np.ones(WINDOW) + 0.05 * rng.standard_normal(WINDOW)),
    "Setor D (bimodal)":       normalize(np.exp(-((t - 1) ** 2) / (2 * 0.8 ** 2)) +
                                     0.8 * np.exp(-((t - 5) ** 2) / (2 * 0.8 ** 2))),
    "Setor E (pico central)":  normalize(np.exp(-((t - 3) ** 2) / (2 * 1.1 ** 2))),
    "Setor F (assimetrico)":   normalize(np.exp(-((t - 2) ** 2) / (2 * 1.6 ** 2)) * (1 + 0.15 * t)),
}
sector_names = list(shapes.keys())
n_sec = len(sector_names)
sector_names, n_sec

In [ ]:
# relative volume weight per sector (kept identical in both scenarios)
volumes = np.array([180, 150, 90, 130, 200, 110])  # arbitrary "K pedidos" units
volumes = dict(zip(sector_names, volumes))
volumes

## 2. Placement of each sector's window

In [ ]:
# As-Is: reactive/historic placement -> most sectors bunched into Block 2 (days 7-13)
as_is_start = {
    "Setor A (pico inicial)": 6,
    "Setor B (pico final)":   8,
    "Setor C (uniforme)":     7,
    "Setor D (bimodal)":      9,
    "Setor E (pico central)": 7,
    "Setor F (assimetrico)":  10,
}

# To-Be: optimizer spreads the SAME shapes across Blocks 1/2/3 so the sum levels out
# (found by local search minimizing std of the total curve, shapes held fixed)
to_be_start = {
    "Setor A (pico inicial)": 0,     # Block 1
    "Setor B (pico final)":  14,     # Block 3
    "Setor C (uniforme)":     5,     # Block 1/2 boundary
    "Setor D (bimodal)":      3,     # Block 1
    "Setor E (pico central)": 13,    # Block 3
    "Setor F (assimetrico)":  9,     # Block 2
}

def place(shape, start):
    curve = np.zeros(N_DAYS)
    for i, v in enumerate(shape):
        day = start + i
        if 0 <= day < N_DAYS:
            curve[day] += v
    return curve

def build_scenario(starts):
    curves = {}
    for name in sector_names:
        curves[name] = place(shapes[name], starts[name]) * volumes[name]
    total = np.sum(list(curves.values()), axis=0)
    return curves, total

curves_as_is, total_as_is = build_scenario(as_is_start)
curves_to_be, total_to_be = build_scenario(to_be_start)

capacity = 100.0  # dashed capacity line, same units as volumes

print(f"As-Is total: mean={
    total_as_is.mean():.1f} std={total_as_is.std():.1f} peak={total_as_is.max():.1f}")
print(f"To-Be total: mean={
    total_to_be.mean():.1f} std={total_to_be.std():.1f} peak={total_to_be.max():.1f}")


## 3. Render: two 3D ridge panels + summed total curve at the front

In [ ]:
palette = ["#4C78A8", "#F58518", "#54A24B", "#E45756", "#72B7B2", "#B279A2"]
block_edges = [0, 7, 14, 21]
block_labels = ["Bloco 1", "Bloco 2", "Bloco 3"]

In [ ]:
fig = plt.figure(figsize=(17, 9), facecolor="white")

GLOBAL_MAX_H = max(
    max(max(c.max() for c in curves_as_is.values()), total_as_is.max()),
    max(max(c.max() for c in curves_to_be.values()), total_to_be.max()),
) * 1.15

def draw_panel(ax, curves, total, title, cap_note):
    ax.set_facecolor("white")
    depth_gap = 1.4
    max_h = GLOBAL_MAX_H
    y_total = n_sec * depth_gap + 0.6
    y_max = y_total + 0.8

    # faint vertical block-boundary planes (full height, semi-transparent)
    for edge in block_edges:
        plane = [[(edge, 0, 0), (edge, y_max, 0), (edge, y_max, max_h), (edge, 0, max_h)]]
        ax.add_collection3d(Poly3DCollection(plane, facecolor="#cccccc", alpha=0.25,
                                              edgecolor="#999999", linewidth=0.6))
    for i, lbl in enumerate(block_labels):
        xm = (block_edges[i] + block_edges[i + 1]) / 2
        ax.text(xm, y_max + 0.3, 0, lbl, color="#777777", fontsize=9, ha="center")

    # sector ridges, one per depth row (y), area curve in x-z plane
    for row, name in enumerate(sector_names):
        y0 = row * depth_gap
        z = curves[name]
        poly_pts = list(zip(
            DAYS, [y0] * N_DAYS, z)) + list(zip(DAYS[::-1], [y0] * N_DAYS, [0] * N_DAYS))
        poly = Poly3DCollection([poly_pts], facecolor=mcolors.to_rgba(palette[row], 0.6),
                                 edgecolor=palette[row], linewidth=1.3)
        ax.add_collection3d(poly)

    # total demand curve, one row further back, bold
    total_pts = list(zip(
        DAYS, [y_total] * N_DAYS, total)) + list(zip(DAYS[::-1], [y_total] * N_DAYS, [0] * N_DAYS))
    poly_total = Poly3DCollection([total_pts], facecolor=mcolors.to_rgba("#333333", 0.4),
                                   edgecolor="#222222", linewidth=2.2)
    ax.add_collection3d(poly_total)

    # capacity dashed line, drawn right over the total ridge only
    ax.plot(
        DAYS, [y_total] * N_DAYS, 
        [capacity] * N_DAYS, 
        linestyle="--", 
        color="#C0392B", 
        linewidth=1.8
        )

    ax.set_xlim(0, N_DAYS)
    ax.set_ylim(0, y_max)
    ax.set_zlim(0, max_h)
    ax.set_xlabel("Dia do ciclo (21 dias)", fontsize=9, labelpad=8)
    ax.set_yticks([i * depth_gap for i in range(n_sec)] + [y_total])
    ax.set_yticklabels([s.split(" (")[0] for s in sector_names] + ["Total"], fontsize=7.5)
    ax.set_zlabel("Volume de pedidos", fontsize=9, labelpad=2)
    ax.set_title(title, fontsize=13, fontweight="bold", pad=12)
    ax.text2D(0.02, 0.0, cap_note, transform=ax.transAxes, fontsize=8.5, color="#C0392B")
    ax.view_init(elev=20, azim=-58)
    ax.xaxis.pane.set_alpha(0.0)
    ax.yaxis.pane.set_alpha(0.0)
    ax.zaxis.pane.set_alpha(0.0)
    ax.grid(False)


In [ ]:
ax1 = fig.add_subplot(1, 2, 1, projection="3d")
draw_panel(ax1, curves_as_is, total_as_is,
           "As-Is — setores concentrados no Bloco 2",
           f"pico = {total_as_is.max():.0f}  >  capacidade ({capacity:.0f})   |   desvio-padrão = {
            total_as_is.std():.0f}")

ax2 = fig.add_subplot(1, 2, 2, projection="3d")
draw_panel(ax2, curves_to_be, total_to_be,
           "To-Be — setores realocados, demanda nivelada",
           f"pico = {total_to_be.max():.0f}  ≤  capacidade ({capacity:.0f})   |   desvio-padrão = {
            total_to_be.std():.0f}")

fig.suptitle("Simulador 2.0 — Nivelamento de Demanda por Realocação de Setores entre Blocos",
             fontsize=16, fontweight="bold", y=0.99)
fig.text(0.5, 0.02,
         "Cada setor preserva sua curva histórica de decisão de compra; apenas a posição muda.",
         ha="center", fontsize=10.5, color="#555555")

plt.subplots_adjust(left=0.02, right=0.98, top=0.88, bottom=0.07, wspace=0.05)

plt.show()